In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.weightstats import ttest_ind
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.stats.api as sms
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error)
from functools import reduce
import re

from xgboost import XGBRegressor

In [ ]:
predictor_cols = [
    "co_wrk_aux",
    "no2_wrk_aux",
    "no_wrk_aux",
    "o3_wrk_aux",
    "rh",
    "temp"
]

sensor_to_site = {
    "Sensor03": "dpw",
    "Sensor08": "dpw",
    "Sensor14": "dpw",
    "Sensor16": "dpw",
    "Sensor18": "dpw",
    "Sensor21": "dpw",

    "Sensor17": "mjf",

    "Sensor05": "pema",
    "Sensor06": "pema",
    "Sensor07": "pema",
    "Sensor15": "pema",
    "Sensor20": "pema",
    "Sensor24": "pema",

    "Sensor02": "pha",
    "Sensor04": "pha",
    "Sensor10": "pha",
    "Sensor12": "pha",
    "Sensor19": "pha",
    "Sensor22": "pha",
    "Sensor23": "pha",
}

results = []
prediction_dfs = []

data_dir = "ML ready NC dpp no2 datasets"

#main loop
for sensor, site in sensor_to_site.items():

    train_path = f"{data_dir}/{sensor}train.csv"
    test_path = f"{data_dir}/{sensor}test.csv"

    #make dataframes
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df["datetime_utc"] = pd.to_datetime(train_df["datetime_utc"])
    test_df["datetime_utc"] = pd.to_datetime(test_df["datetime_utc"])

    target_col = f"{site}_quant_no2"

    X_train = train_df[predictor_cols]
    y_train = train_df[target_col]

    X_test = test_df[predictor_cols]
    y_test = test_df[target_col]

    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    #grab importance values from the training of each algorithm
    importance = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_
    })

    importance = importance.sort_values("importance", ascending=False)

    top1_feature = importance.iloc[0]["feature"]
    top1_importance = importance.iloc[0]["importance"]

    top2_feature = importance.iloc[1]["feature"]
    top2_importance = importance.iloc[1]["importance"]

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #start compiling dataframe of all calibrated data
    pred_df = pd.DataFrame({
        "datetime_utc": test_df["datetime_utc"],
        sensor: y_test_pred
    })

    prediction_dfs.append(pred_df)

    train_r2 = r2_score(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    results.append({
        "sensor": sensor,
        "site": site,

        "train_r2": train_r2,
        "train_mae": train_mae,
        "train_rmse": train_rmse,

        "test_r2": test_r2,
        "test_mae": test_mae,
        "test_rmse": test_rmse,

        "top1_feature": top1_feature,
        "top1_importance": top1_importance,

        "top2_feature": top2_feature,
        "top2_importance": top2_importance,

        "n_train": len(train_df),
        "n_test": len(test_df)
    })

    print(f"{sensor} ({site}) done")

results_df = pd.DataFrame(results)

results_df = results_df.round({
    "train_r2": 3,
    "test_r2": 3,
    "train_mae": 2,
    "test_mae": 2,
    "train_rmse": 2,
    "test_rmse": 2,
    "top1_importance": 3,
    "top2_importance": 3
})

results_df = results_df.sort_values("test_r2", ascending=False)

results_df

Sensor03 (dpw) done
Sensor08 (dpw) done
Sensor14 (dpw) done
Sensor16 (dpw) done
Sensor18 (dpw) done
Sensor21 (dpw) done
Sensor17 (mjf) done
Sensor05 (pema) done
Sensor06 (pema) done
Sensor07 (pema) done
Sensor15 (pema) done
Sensor20 (pema) done
Sensor24 (pema) done
Sensor02 (pha) done
Sensor04 (pha) done
Sensor10 (pha) done
Sensor12 (pha) done
Sensor19 (pha) done
Sensor22 (pha) done
Sensor23 (pha) done


,sensor,site,train_r2,train_mae,train_rmse,test_r2,test_mae,test_rmse,top1_feature,top1_importance,top2_feature,top2_importance,n_train,n_test
16,Sensor12,pha,0.996,0.28,0.39,0.867,1.58,2.38,no2_wrk_aux,0.573,temp,0.224,521,5613
13,Sensor02,pha,0.996,0.31,0.40,0.860,1.69,2.44,no2_wrk_aux,0.530,temp,0.248,521,5613
6,Sensor17,mjf,0.991,0.46,0.60,0.856,1.50,2.15,no2_wrk_aux,0.486,temp,0.238,520,5606
14,Sensor04,pha,0.994,0.37,0.50,0.854,1.75,2.50,no2_wrk_aux,0.537,temp,0.294,521,5587
18,Sensor22,pha,0.995,0.29,0.38,0.845,1.66,2.43,no2_wrk_aux,0.561,temp,0.156,464,4889
15,Sensor10,pha,0.996,0.31,0.42,0.797,2.02,3.01,no2_wrk_aux,0.475,temp,0.257,460,4763
19,Sensor23,pha,0.994,0.39,0.53,0.721,2.50,3.59,temp,0.533,rh,0.156,406,4607
10,Sensor15,pema,0.994,0.37,0.49,0.672,2.46,3.42,no2_wrk_aux,0.549,temp,0.221,464,4754
9,Sensor07,pema,0.989,0.48,0.66,0.666,2.51,3.45,no2_wrk_aux,0.414,temp,0.289,521,5197
5,Sensor21,dpw,0.994,0.29,0.38,0.638,2.03,2.93,no2_wrk_aux,0.533,temp,0.153,395,4082


In [ ]:
#compile predictions into one dataframe

all_predictions = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on="datetime_utc",
        how="outer"
    ),
    prediction_dfs
)

all_predictions = (
    all_predictions
    .sort_values("datetime_utc")
    .reset_index(drop=True)
)

sensor_cols = all_predictions.columns.drop("datetime_utc")
all_predictions[sensor_cols] = all_predictions[sensor_cols].round(2)

all_predictions.to_csv(
    "calibrated datasets/NC DPP sensors calibrated.csv",
    index=False
)

all_predictions.head()

,datetime_utc,Sensor03,Sensor08,Sensor14,Sensor16,Sensor18,Sensor21,Sensor17,Sensor05,Sensor06,...,Sensor15,Sensor20,Sensor24,Sensor02,Sensor04,Sensor10,Sensor12,Sensor19,Sensor22,Sensor23
0,2025-10-09 22:00:00,31.219999,28.770000,26.820000,29.950001,35.320000,24.389999,6.870000,NaN,NaN,...,NaN,NaN,NaN,34.310001,33.509998,30.500000,29.170000,14.430000,28.530001,27.270000
1,2025-10-09 23:00:00,33.639999,31.209999,30.879999,32.360001,36.320000,25.270000,11.550000,28.180000,31.190001,...,31.940001,33.590000,32.680000,31.000000,32.139999,31.950001,32.110001,16.639999,30.280001,29.520000
2,2025-10-10 00:00:00,29.850000,31.830000,33.360001,25.090000,34.110001,31.010000,16.930000,28.879999,30.450001,...,30.549999,31.139999,31.510000,30.170000,30.760000,31.020000,30.540001,27.240000,30.730000,28.010000
3,2025-10-10 01:00:00,29.660000,30.660000,30.879999,27.049999,30.059999,31.360001,16.459999,23.379999,26.730000,...,28.530001,29.680000,28.360001,30.639999,28.299999,29.459999,24.459999,24.410000,26.990000,24.590000
4,2025-10-10 02:00:00,28.400000,28.040001,28.490000,27.719999,28.900000,28.760000,17.480000,21.950001,26.860001,...,27.260000,30.280001,24.910000,28.170000,27.370001,26.450001,22.450001,23.459999,20.410000,24.540001


In [ ]:
#combine calibrated dataframes into one dataframe
#YOU MUST RUN COLLOCATED MODELS FIRST

c_df = pd.read_csv("calibrated datasets/C DPP sensors calibrated.csv")
nc_df = pd.read_csv("calibrated datasets/NC DPP sensors calibrated.csv")

c_df["datetime_utc"] = pd.to_datetime(c_df["datetime_utc"])
nc_df["datetime_utc"] = pd.to_datetime(nc_df["datetime_utc"])

combined = pd.merge(
    c_df,
    nc_df,
    on="datetime_utc",
    how="outer"
)

if "Sensor11" not in combined.columns:
    combined["Sensor11"] = np.nan

def sensor_sort_key(col):
    if col == "datetime_utc":
        return -1
    match = re.search(r"Sensor(\d+)", col)
    if match:
        return int(match.group(1))
    return 9999

sorted_cols = sorted(combined.columns, key=sensor_sort_key)
combined = combined[sorted_cols]

combined = combined.sort_values("datetime_utc").reset_index(drop=True)

combined.to_csv(
    "calibrated datasets/DPP sensors calibrated.csv",
    index=False
)

combined.head()

,datetime_utc,Sensor01,Sensor02,Sensor03,Sensor04,Sensor05,Sensor06,Sensor07,Sensor08,Sensor09,...,Sensor16,Sensor17,Sensor18,Sensor19,Sensor20,Sensor21,Sensor22,Sensor23,Sensor24,Sensor25
0,2025-10-09 22:00:00,7.96,34.31,31.22,33.51,NaN,NaN,NaN,28.77,NaN,...,29.95,6.87,35.32,14.43,NaN,24.39,28.53,27.27,NaN,NaN
1,2025-10-09 23:00:00,12.42,31.00,33.64,32.14,28.18,31.19,30.65,31.21,NaN,...,32.36,11.55,36.32,16.64,33.59,25.27,30.28,29.52,32.68,29.41
2,2025-10-10 00:00:00,17.04,30.17,29.85,30.76,28.88,30.45,28.62,31.83,NaN,...,25.09,16.93,34.11,27.24,31.14,31.01,30.73,28.01,31.51,29.95
3,2025-10-10 01:00:00,18.74,30.64,29.66,28.30,23.38,26.73,27.44,30.66,NaN,...,27.05,16.46,30.06,24.41,29.68,31.36,26.99,24.59,28.36,23.46
4,2025-10-10 02:00:00,20.62,28.17,28.40,27.37,21.95,26.86,25.20,28.04,NaN,...,27.72,17.48,28.90,23.46,30.28,28.76,20.41,24.54,24.91,21.85
